# 🔌 CircuitGPT — AI-Powered Circuit Analysis

**A complete end-to-end pipeline for intelligent circuit board analysis using Computer Vision and Generative AI.**

---

## 📋 Project Overview

CircuitGPT combines three specialized AI models with a Generative AI backbone to:

| # | Capability | Model / Technology |
|---|------------|--------------------|
| 1 | **Classify** 37 types of electronic components | ResNet18 (Transfer Learning) |
| 2 | **Detect** IoT boards & sensors on breadboards | YOLOv8 (Object Detection) |
| 3 | **Segment** wires and cables | U-Net + ResNet34 Encoder |
| 4 | **Explain & Debug** circuits in natural language | Google Gemini (LLM) |

---

### AI Technologies Used

**Computer Vision (OpenCV + CNN + YOLOv8)**
- YOLOv8 detects circuit components in real time — resistors, capacitors, ICs, wires — using bounding boxes
- OpenCV performs image preprocessing, edge detection, and contour detection
- CNN classifies circuit conditions (correct wiring, short circuits, polarity errors)

**Generative AI (Large Language Models)**
- Explains circuit behaviour in simple, readable language
- Suggests corrections and troubleshooting steps based on detected faults
- Converts natural language questions into intelligent circuit guidance

---

### Main Objectives
1. Develop a Visual AI system that understands circuit diagrams
2. Detect wiring errors and component faults automatically
3. Provide intelligent troubleshooting suggestions
4. Convert natural language input into circuit explanations
5. Improve efficiency in hardware debugging

In [ ]:
import os
import sys
import json
import glob
import shutil
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageDraw
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

from pycocotools.coco import COCO
from ultralytics import YOLO
import yaml

from sklearn.metrics import classification_report, confusion_matrix

%matplotlib inline

print('✅ All libraries imported successfully!')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ZIP_PATH = '/content/drive/MyDrive/Datasets.zip'
EXTRACT_DIR = '/content/CircuitGPT'

if not os.path.exists(os.path.join(EXTRACT_DIR, 'Datasets')):
    print(f'📦  Extracting {ZIP_PATH} ...')
    os.makedirs(EXTRACT_DIR, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(EXTRACT_DIR)
    print('✅ Extraction complete!')
else:
    print('✅ Dataset already extracted.')

# ── Set base paths ──
BASE_DIR       = EXTRACT_DIR
DATASET_ROOT   = os.path.join(BASE_DIR, 'Datasets')
MODEL_SAVE_DIR = os.path.join(BASE_DIR, 'models')
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

# Dataset sub-paths
IMAGES_DIR = os.path.join(DATASET_ROOT, 'images')
COCO_DIR   = os.path.join(DATASET_ROOT, 'IoTKITs', '32.v1i.coco')
CABLE_DIR  = os.path.join(DATASET_ROOT, 'cables_dataset-master')

print(f'\nBASE_DIR     : {BASE_DIR}')
print(f'IMAGES_DIR   : {IMAGES_DIR}  — exists: {os.path.isdir(IMAGES_DIR)}')
print(f'COCO_DIR     : {COCO_DIR}  — exists: {os.path.isdir(COCO_DIR)}')
print(f'CABLE_DIR    : {CABLE_DIR}  — exists: {os.path.isdir(CABLE_DIR)}')
print(f'MODEL_SAVE   : {MODEL_SAVE_DIR}')

In [ ]:
if os.path.isdir(IMAGES_DIR):
    classes = sorted([d for d in os.listdir(IMAGES_DIR)
    if os.path.isdir(os.path.join(IMAGES_DIR, d)) and d != 'images'])
    total_imgs = sum(len(os.listdir(os.path.join(IMAGES_DIR, c))) for c in classes)
    print(f'Component classes : {len(classes)}')
    print(f'Total images      : {total_imgs}')
    print(f'Classes: {classes[:10]} ... ({len(classes)} total)')
else:
    print('images/ directory not found!')

for subset in ['cable_dataset_simple', 'cable_dataset_hard']:
    p = os.path.join(CABLE_DIR, subset)
    if os.path.isdir(p):
        n = len([f for f in os.listdir(p) if f.endswith('.jpg')])
        print(f'🔌  {subset}: {n} images')

for split in ['train', 'valid']:
    ann = os.path.join(COCO_DIR, split, '_annotations.coco.json')
    if os.path.exists(ann):
        with open(ann) as f:
            data = json.load(f)
        print(f'IoTKITs {split}: {len(data["images"])} images, '
              f'{len(data["annotations"])} annotations, '
              f'{len(data["categories"])} categories')

---
## 3 · Model 1 — Electronic Component Classifier (ResNet18)

Transfer-learning approach using a pre-trained ResNet18 to classify **37 types** of electronic components (LED, transistor, relay, capacitor, etc.).

| Aspect | Detail |
|--------|--------|
| Architecture | ResNet18 + custom FC head |
| Input size | 224 × 224 |
| Pre-training | ImageNet |
| Strategy | Freeze early layers, fine-tune last 20 params |
| Augmentation | Flip, rotation, colour jitter, affine |

In [ ]:
CLF_IMAGE_SIZE   = 224
CLF_BATCH_SIZE   = 32
CLF_NUM_EPOCHS   = 25
CLF_LEARNING_RATE = 1e-3
CLF_TRAIN_SPLIT  = 0.8
CLF_NUM_WORKERS  = 2

print('Classifier config set.')

In [ ]:
class ComponentDataset(Dataset):
    """
    Loads electronic component images from directory structure:
        images/
            LED/
                LED045.jpg
            transistor/
                ...
    """
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.samples = []
        self.classes = []
        self.class_to_idx = {}

        subdirs = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d)) and d != 'images'
        ])
        self.classes = subdirs
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(subdirs)}

        valid_ext = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
        for cls_name in subdirs:
            cls_dir = os.path.join(root_dir, cls_name)
            for fname in os.listdir(cls_dir):
                if os.path.splitext(fname)[1].lower() in valid_ext:
                    self.samples.append((
                        os.path.join(cls_dir, fname),
                        self.class_to_idx[cls_name]
                    ))

        print(f'[ComponentDataset] Found {len(self.samples)} images '
              f'across {len(self.classes)} classes')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

print('ComponentDataset class defined.')

In [ ]:
clf_train_transform = transforms.Compose([
    transforms.Resize((CLF_IMAGE_SIZE, CLF_IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3,
                           saturation=0.2, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

clf_val_transform = transforms.Compose([
    transforms.Resize((CLF_IMAGE_SIZE, CLF_IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

print('Transforms defined.')

In [ ]:
full_dataset = ComponentDataset(IMAGES_DIR, transform=None)
num_classes  = len(full_dataset.classes)

total      = len(full_dataset)
train_size = int(CLF_TRAIN_SPLIT * total)
val_size   = total - train_size

train_subset, val_subset = random_split(
    full_dataset, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        img_path, label = self.subset.dataset.samples[self.subset.indices[idx]]
        image = Image.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, label

train_dataset = TransformSubset(train_subset, clf_train_transform)
val_dataset   = TransformSubset(val_subset, clf_val_transform)

train_loader = DataLoader(train_dataset, batch_size=CLF_BATCH_SIZE,
                          shuffle=True, num_workers=CLF_NUM_WORKERS,
                          pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=CLF_BATCH_SIZE,
                          shuffle=False, num_workers=CLF_NUM_WORKERS,
                          pin_memory=True)

print(f'Classes       : {num_classes}')
print(f'Train samples : {len(train_dataset)}')
print(f'Val samples   : {len(val_dataset)}')

In [ ]:
def build_classifier(num_classes):
    """ResNet18 with custom classification head."""
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

    # Freeze early layers
    for param in list(model.parameters())[:-20]:
        param.requires_grad = False

    num_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(num_features, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.2),
        nn.Linear(256, num_classes)
    )
    return model.to(DEVICE)

clf_model = build_classifier(num_classes)

trainable = sum(p.numel() for p in clf_model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in clf_model.parameters())
print(f'Total params     : {total_p:,}')
print(f'Trainable params : {trainable:,}')

In [ ]:
def clf_train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(dataloader, desc='  Training', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total


def clf_validate(model, dataloader, criterion):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='  Validating', leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return running_loss / total, 100.0 * correct / total, all_preds, all_labels

print('Training functions defined.')

In [ ]:
clf_criterion = nn.CrossEntropyLoss()
clf_optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, clf_model.parameters()),
    lr=CLF_LEARNING_RATE, weight_decay=1e-4
)
clf_scheduler = optim.lr_scheduler.StepLR(clf_optimizer, step_size=8, gamma=0.1)

best_val_acc = 0.0
clf_train_losses, clf_val_losses = [], []
clf_train_accs, clf_val_accs = [], []

print('Training Component Classifier (ResNet18)')

for epoch in range(1, CLF_NUM_EPOCHS + 1):
    print(f'\n  Epoch {epoch}/{CLF_NUM_EPOCHS}')

    t_loss, t_acc = clf_train_one_epoch(clf_model, train_loader,
                                        clf_criterion, clf_optimizer)
    v_loss, v_acc, v_preds, v_labels = clf_validate(clf_model, val_loader,
                                                     clf_criterion)
    clf_scheduler.step()

    clf_train_losses.append(t_loss)
    clf_val_losses.append(v_loss)
    clf_train_accs.append(t_acc)
    clf_val_accs.append(v_acc)

    print(f'    Train Loss: {t_loss:.4f} | Train Acc: {t_acc:.2f}%')
    print(f'    Val Loss:   {v_loss:.4f} | Val Acc:   {v_acc:.2f}%')

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': clf_model.state_dict(),
            'optimizer_state_dict': clf_optimizer.state_dict(),
            'val_acc': v_acc,
            'classes': full_dataset.classes,
            'class_to_idx': full_dataset.class_to_idx,
        }, os.path.join(MODEL_SAVE_DIR, 'component_classifier_best.pth'))
        print(f'    ★ New best model saved! (Val Acc: {v_acc:.2f}%)')

print(f'\nBest Validation Accuracy: {best_val_acc:.2f}%')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(clf_train_losses, label='Train Loss', linewidth=2)
ax1.plot(clf_val_losses, label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(clf_train_accs, label='Train Acc', linewidth=2)
ax2.plot(clf_val_accs, label='Val Acc', linewidth=2)
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training & Validation Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Model 1 — Component Classifier Training', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'classifier_training_curves.png'), dpi=150)
plt.show()

In [ ]:
report = classification_report(
    v_labels, v_preds,
    target_names=full_dataset.classes,
    zero_division=0
)
print(report)

# Save class mapping
mapping_path = os.path.join(MODEL_SAVE_DIR, 'component_classes.json')
with open(mapping_path, 'w') as f:
    json.dump({
        'classes': full_dataset.classes,
        'class_to_idx': full_dataset.class_to_idx,
        'idx_to_class': {v: k for k, v in full_dataset.class_to_idx.items()}
    }, f, indent=2)
print(f'Class mapping saved to: {mapping_path}')
print(f'Component Classifier training complete!')

---
## 4 · Model 2 — IoT Kit Object Detector (YOLOv8)

Fine-tunes **YOLOv8n** (nano) on the IoTKITs COCO dataset to detect electronic boards, sensors, and components on breadboards.

| Aspect | Detail |
|--------|--------|
| Architecture | YOLOv8 Nano |
| Input size | 640 × 640 |
| Pre-training | COCO |
| Format conversion | COCO → YOLO |
| Patience | 10 epochs (early stopping) |

In [ ]:
DET_IMAGE_SIZE = 640
DET_BATCH_SIZE = 16
DET_NUM_EPOCHS = 50
YOLO_DIR = os.path.join(BASE_DIR, 'datasets_yolo')  # converted dataset output

print('Detector config set.')

In [ ]:
def coco_to_yolo(coco_dir, yolo_dir, split='train'):
    """
    Convert COCO format annotations to YOLO format.
    YOLO: <class_id> <x_center> <y_center> <width> <height>  (all normalised)
    """
    split_dir = os.path.join(coco_dir, split)
    ann_file  = os.path.join(split_dir, '_annotations.coco.json')

    if not os.path.exists(ann_file):
        print(f'Annotation file not found: {ann_file}')
        return [], {}

    coco = COCO(ann_file)

    img_out = os.path.join(yolo_dir, 'images', split)
    lbl_out = os.path.join(yolo_dir, 'labels', split)
    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    categories = coco.loadCats(coco.getCatIds())
    cat_id_to_yolo = {}
    cat_names = []
    for i, cat in enumerate(sorted(categories, key=lambda x: x['id'])):
        cat_id_to_yolo[cat['id']] = i
        cat_names.append(cat['name'])

    print(f'  [{split}] Categories ({len(cat_names)}): {cat_names[:10]}...')

    img_ids = coco.getImgIds()
    converted = 0

    for img_id in img_ids:
        img_info = coco.loadImgs(img_id)[0]
        img_w, img_h = img_info['width'], img_info['height']
        img_fname = img_info['file_name']

        src_path = os.path.join(split_dir, img_fname)
        if os.path.exists(src_path):
            shutil.copy2(src_path, os.path.join(img_out, img_fname))
        else:
            continue

        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)

        label_fname = os.path.splitext(img_fname)[0] + '.txt'
        label_path  = os.path.join(lbl_out, label_fname)

        with open(label_path, 'w') as f:
            for ann in anns:
                if 'bbox' not in ann:
                    continue
                x, y, w, h = ann['bbox']
                x_c = max(0, min(1, (x + w / 2) / img_w))
                y_c = max(0, min(1, (y + h / 2) / img_h))
                w_n = max(0, min(1, w / img_w))
                h_n = max(0, min(1, h / img_h))
                cid = cat_id_to_yolo.get(ann['category_id'], 0)
                f.write(f'{cid} {x_c:.6f} {y_c:.6f} {w_n:.6f} {h_n:.6f}\n')
        converted += 1

    print(f'  [{split}] Converted {converted} images')
    return cat_names, cat_id_to_yolo

print('coco_to_yolo() defined.')

In [ ]:
print('Converting COCO → YOLO format...\n')

cat_names_train, _ = coco_to_yolo(COCO_DIR, YOLO_DIR, split='train')
cat_names_val, _   = coco_to_yolo(COCO_DIR, YOLO_DIR, split='valid')

cat_names = cat_names_train if cat_names_train else cat_names_val

# Create data.yaml
data_config = {
    'path': os.path.abspath(YOLO_DIR),
    'train': 'images/train',
    'val': 'images/valid',
    'nc': len(cat_names),
    'names': cat_names,
}

yaml_path = os.path.join(YOLO_DIR, 'data.yaml')
os.makedirs(YOLO_DIR, exist_ok=True)
with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print(f'\ndata.yaml saved to: {yaml_path}')
print(f'Categories: {len(cat_names)}')

In [ ]:
print('=' * 60)
print('  Training YOLOv8 Nano — IoT Object Detector')
print('=' * 60)

det_model = YOLO('yolov8n.pt')  # pretrained nano model

results = det_model.train(
    data=yaml_path,
    epochs=DET_NUM_EPOCHS,
    imgsz=DET_IMAGE_SIZE,
    batch=DET_BATCH_SIZE,
    name='circuitgpt_detector',
    project=MODEL_SAVE_DIR,
    patience=10,
    save=True,
    plots=True,
    verbose=True,
    device=0 if torch.cuda.is_available() else 'cpu',
)

print('\nYOLOv8 training complete!')

In [ ]:
metrics = det_model.val(data=yaml_path)

print(f'\n─── Detection Results ───')
print(f'  mAP@50      : {metrics.box.map50:.4f}')
print(f'  mAP@50-95   : {metrics.box.map:.4f}')
print(f'  Precision   : {metrics.box.mp:.4f}')
print(f'  Recall      : {metrics.box.mr:.4f}')

# Copy best model
best_src = os.path.join(MODEL_SAVE_DIR, 'circuitgpt_detector', 'weights', 'best.pt')
best_dst = os.path.join(MODEL_SAVE_DIR, 'iot_detector_best.pt')
if os.path.exists(best_src):
    shutil.copy2(best_src, best_dst)
    print(f'\nBest model saved → {best_dst}')

# Save category info
with open(os.path.join(MODEL_SAVE_DIR, 'detector_categories.json'), 'w') as f:
    json.dump({'categories': cat_names, 'num_categories': len(cat_names)}, f, indent=2)

print('IoT Object Detector training complete!')

---
## 5 · Model 3 — Wire / Cable Segmentation (U-Net)

Binary segmentation of wires and cables using **U-Net** with a **ResNet34** encoder pre-trained on ImageNet.

| Aspect | Detail |
|--------|--------|
| Architecture | U-Net (segmentation_models_pytorch) |
| Encoder | ResNet34 (ImageNet) |
| Input size | 256 × 256 |
| Loss | BCE + Dice Loss |
| Metrics | IoU, Dice Score |

In [ ]:
SEG_IMAGE_SIZE   = 256
SEG_BATCH_SIZE   = 8
SEG_NUM_EPOCHS   = 30
SEG_LEARNING_RATE = 1e-4
SEG_TRAIN_SPLIT  = 0.8
SEG_NUM_WORKERS  = 2

print('Segmentation config set.')

In [ ]:
class CableDataset(Dataset):
    """
    Loads cable images and binary segmentation masks.
    Expects: <id>.jpg  +  <id>_mask_all.png
    """
    def __init__(self, data_dirs, transform=None):
        self.transform = transform
        self.samples = []

        for data_dir in data_dirs:
            if not os.path.exists(data_dir):
                print(f'  ⚠ Directory not found: {data_dir}')
                continue
            image_files = sorted(glob.glob(os.path.join(data_dir, '*.jpg')))
            for img_path in image_files:
                basename = os.path.splitext(os.path.basename(img_path))[0]
                mask_path = os.path.join(data_dir, f'{basename}_mask_all.png')
                if os.path.exists(mask_path):
                    self.samples.append((img_path, mask_path))

        print(f'[CableDataset] Found {len(self.samples)} image-mask pairs')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, mask_path = self.samples[idx]
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)

        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']

        if isinstance(mask, torch.Tensor):
            mask = mask.unsqueeze(0)
        else:
            mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        return image, mask

print('CableDataset class defined.')

In [ ]:
seg_train_transform = A.Compose([
    A.Resize(SEG_IMAGE_SIZE, SEG_IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.3),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15,
                       rotate_limit=15, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2,
                               contrast_limit=0.2, p=0.4),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

seg_val_transform = A.Compose([
    A.Resize(SEG_IMAGE_SIZE, SEG_IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# Build datasets
cable_data_dirs = [
    os.path.join(CABLE_DIR, 'cable_dataset_simple'),
    os.path.join(CABLE_DIR, 'cable_dataset_hard'),
]
seg_full_dataset = CableDataset(cable_data_dirs, transform=None)

seg_total = len(seg_full_dataset)
seg_train_size = int(SEG_TRAIN_SPLIT * seg_total)
seg_val_size   = seg_total - seg_train_size

seg_train_idx, seg_val_idx = random_split(
    range(seg_total), [seg_train_size, seg_val_size],
    generator=torch.Generator().manual_seed(42)
)

class SegSubset(Dataset):
    def __init__(self, base_dataset, indices, transform):
        self.base = base_dataset
        self.indices = list(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, idx):
        img_path, mask_path = self.base.samples[self.indices[idx]]
        image = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = (mask > 127).astype(np.float32)
        if self.transform:
            t = self.transform(image=image, mask=mask)
            image, mask = t['image'], t['mask']
        if isinstance(mask, torch.Tensor):
            mask = mask.unsqueeze(0)
        else:
            mask = torch.tensor(mask, dtype=torch.float32).unsqueeze(0)
        return image, mask

seg_train_ds = SegSubset(seg_full_dataset, seg_train_idx, seg_train_transform)
seg_val_ds   = SegSubset(seg_full_dataset, seg_val_idx, seg_val_transform)

seg_train_loader = DataLoader(seg_train_ds, batch_size=SEG_BATCH_SIZE,
                              shuffle=True, num_workers=SEG_NUM_WORKERS,
                              pin_memory=True)
seg_val_loader   = DataLoader(seg_val_ds, batch_size=SEG_BATCH_SIZE,
                              shuffle=False, num_workers=SEG_NUM_WORKERS,
                              pin_memory=True)

print(f'Train samples : {len(seg_train_ds)}')
print(f'Val samples   : {len(seg_val_ds)}')

In [ ]:
class DiceBCELoss(nn.Module):
    """Combined Binary Cross-Entropy + Dice Loss."""
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, pred, target):
        bce_loss = self.bce(pred, target)
        pred_sigmoid = torch.sigmoid(pred)
        pred_flat = pred_sigmoid.view(-1)
        target_flat = target.view(-1)
        intersection = (pred_flat * target_flat).sum()
        dice = (2.0 * intersection + self.smooth) / (
            pred_flat.sum() + target_flat.sum() + self.smooth)
        return bce_loss + (1.0 - dice)


def compute_iou(pred, target, threshold=0.5):
    pred_bin = (torch.sigmoid(pred) > threshold).float()
    inter = (pred_bin * target).sum()
    union = pred_bin.sum() + target.sum() - inter
    return ((inter + 1e-7) / (union + 1e-7)).item()


def compute_dice(pred, target, threshold=0.5):
    pred_bin = (torch.sigmoid(pred) > threshold).float()
    inter = (pred_bin * target).sum()
    return ((2.0 * inter + 1e-7) / (pred_bin.sum() + target.sum() + 1e-7)).item()


# Build U-Net
seg_model = smp.Unet(
    encoder_name='resnet34',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None,
).to(DEVICE)

print(f'U-Net built (encoder: ResNet34)')

In [ ]:
def seg_train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    total_loss, total_iou, count = 0.0, 0.0, 0
    for images, masks in tqdm(dataloader, desc='  Training', leave=False):
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_iou += compute_iou(outputs, masks)
        count += 1
    return total_loss / count, total_iou / count


def seg_validate(model, dataloader, criterion):
    model.eval()
    total_loss, total_iou, total_dice, count = 0.0, 0.0, 0.0, 0
    with torch.no_grad():
        for images, masks in tqdm(dataloader, desc='  Validating', leave=False):
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()
            total_iou += compute_iou(outputs, masks)
            total_dice += compute_dice(outputs, masks)
            count += 1
    return total_loss / count, total_iou / count, total_dice / count

print('Segmentation training functions defined.')

In [ ]:
seg_criterion = DiceBCELoss()
seg_optimizer = optim.Adam(seg_model.parameters(),
                           lr=SEG_LEARNING_RATE, weight_decay=1e-4)
seg_scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    seg_optimizer, mode='min', factor=0.5, patience=5, verbose=True)

best_val_iou = 0.0
seg_train_losses, seg_val_losses, seg_val_ious = [], [], []

print('=' * 60)
print('  Training Wire Segmentation (U-Net + ResNet34)')
print('=' * 60)

for epoch in range(1, SEG_NUM_EPOCHS + 1):
    print(f'\n  Epoch {epoch}/{SEG_NUM_EPOCHS}')

    t_loss, t_iou = seg_train_one_epoch(seg_model, seg_train_loader,
                                         seg_criterion, seg_optimizer)
    v_loss, v_iou, v_dice = seg_validate(seg_model, seg_val_loader,
                                          seg_criterion)
    seg_scheduler.step(v_loss)

    seg_train_losses.append(t_loss)
    seg_val_losses.append(v_loss)
    seg_val_ious.append(v_iou)

    print(f'    Train Loss: {t_loss:.4f} | Train IoU: {t_iou:.4f}')
    print(f'    Val Loss:   {v_loss:.4f} | Val IoU:   {v_iou:.4f} '
          f'| Val Dice: {v_dice:.4f}')

    if v_iou > best_val_iou:
        best_val_iou = v_iou
        torch.save({
            'epoch': epoch,
            'model_state_dict': seg_model.state_dict(),
            'optimizer_state_dict': seg_optimizer.state_dict(),
            'val_iou': v_iou,
            'val_dice': v_dice,
        }, os.path.join(MODEL_SAVE_DIR, 'wire_segmentation_best.pth'))
        print(f'    ★ New best model saved! (IoU: {v_iou:.4f})')

print(f'\nBest Validation IoU: {best_val_iou:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(seg_train_losses, label='Train Loss', linewidth=2)
ax1.plot(seg_val_losses, label='Val Loss', linewidth=2)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Training & Validation Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(seg_val_ious, label='Val IoU', linewidth=2, color='green')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('IoU')
ax2.set_title('Validation IoU'); ax2.legend(); ax2.grid(True, alpha=0.3)

plt.suptitle('Model 3 — Wire Segmentation Training', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'segmentation_training_curves.png'), dpi=150)
plt.show()

In [ ]:
seg_model.eval()
num_vis = min(6, len(seg_val_ds))
fig, axes = plt.subplots(num_vis, 3, figsize=(12, 4 * num_vis))

indices = np.random.choice(len(seg_val_ds), num_vis, replace=False)

for i, idx in enumerate(indices):
    image, mask = seg_val_ds[idx]
    with torch.no_grad():
        pred = seg_model(image.unsqueeze(0).to(DEVICE))
        pred_mask = (torch.sigmoid(pred) > 0.5).float()

    # Denormalize
    img_disp = image.cpu().numpy().transpose(1, 2, 0)
    mean = np.array([0.485, 0.456, 0.406])
    std  = np.array([0.229, 0.224, 0.225])
    img_disp = np.clip(std * img_disp + mean, 0, 1)

    axes[i, 0].imshow(img_disp);          axes[i, 0].set_title('Input');      axes[i, 0].axis('off')
    axes[i, 1].imshow(mask.squeeze().cpu().numpy(), cmap='gray')
    axes[i, 1].set_title('Ground Truth'); axes[i, 1].axis('off')
    axes[i, 2].imshow(pred_mask.squeeze().cpu().numpy(), cmap='gray')
    axes[i, 2].set_title('Predicted');    axes[i, 2].axis('off')

plt.suptitle('Wire Segmentation — Sample Predictions', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(MODEL_SAVE_DIR, 'segmentation_predictions.png'), dpi=150)
plt.show()
print('Wire Segmentation training complete!')

---
## 6 · CircuitGPT

Load all three trained models and run the **complete CircuitGPT analysis** on any circuit / breadboard image:
1. **Classify** the dominant component
2. **Detect** all IoT components with bounding boxes
3. **Segment** wires and compute coverage
4. **Visualise** everything in a single figure

In [ ]:
print('Loading trained models...\n')

clf_ckpt_path = os.path.join(MODEL_SAVE_DIR, 'component_classifier_best.pth')
clf_classes_path = os.path.join(MODEL_SAVE_DIR, 'component_classes.json')

inf_classifier, inf_class_info = None, None
if os.path.exists(clf_ckpt_path):
    with open(clf_classes_path) as f:
        inf_class_info = json.load(f)
    n_cls = len(inf_class_info['classes'])
    inf_classifier = models.resnet18(weights=None)
    inf_classifier.fc = nn.Sequential(
        nn.Dropout(0.4), nn.Linear(inf_classifier.fc.in_features, 256),
        nn.ReLU(inplace=True), nn.Dropout(0.2), nn.Linear(256, n_cls))
    inf_classifier.load_state_dict(
        torch.load(clf_ckpt_path, map_location=DEVICE)['model_state_dict'])
    inf_classifier.to(DEVICE).eval()
    print(f'Component Classifier loaded ({n_cls} classes)')
else:
    print('Classifier checkpoint not found.')

det_path = os.path.join(MODEL_SAVE_DIR, 'iot_detector_best.pt')
inf_detector = None
if os.path.exists(det_path):
    inf_detector = YOLO(det_path)
    print('IoT Object Detector loaded')
else:
    print('Detector checkpoint not found.')

seg_ckpt_path = os.path.join(MODEL_SAVE_DIR, 'wire_segmentation_best.pth')
inf_segmentor = None
if os.path.exists(seg_ckpt_path):
    inf_segmentor = smp.Unet(
        encoder_name='resnet34', encoder_weights=None,
        in_channels=3, classes=1, activation=None)
    inf_segmentor.load_state_dict(
        torch.load(seg_ckpt_path, map_location=DEVICE)['model_state_dict'])
    inf_segmentor.to(DEVICE).eval()
    print('Wire Segmentation model loaded')
else:
    print('Segmentation checkpoint not found.')

print('\nModel loading complete.')

In [ ]:
def classify_component(model, class_info, image_path):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    image = Image.open(image_path).convert('RGB')
    img_t = transform(image).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        out = model(img_t)
        probs = torch.softmax(out, dim=1)
        conf, pred = probs.max(1)

    idx_to_class = class_info.get('idx_to_class', {})
    cls_name = idx_to_class.get(str(pred.item()), f'class_{pred.item()}')

    top5_p, top5_i = probs.topk(5, dim=1)
    top5 = [{'class': idx_to_class.get(str(i.item()), f'class_{i.item()}'),
             'confidence': f'{p.item()*100:.2f}%'}
            for p, i in zip(top5_p[0], top5_i[0])]

    return {'predicted_class': cls_name,
            'confidence': f'{conf.item()*100:.2f}%',
            'top_5_predictions': top5}


def detect_components(model, image_path):
    """Detect IoT components using YOLOv8."""
    results = model.predict(source=image_path, conf=0.25, iou=0.45, verbose=False)
    detections = []
    if results:
        for box in results[0].boxes:
            cls_id = int(box.cls[0])
            detections.append({
                'class': results[0].names.get(cls_id, f'class_{cls_id}'),
                'confidence': f'{float(box.conf[0])*100:.2f}%',
                'bbox': [round(v, 1) for v in box.xyxy[0].cpu().numpy().tolist()],
            })
    return detections


def segment_wires(model, image_path):
    """Segment wires using U-Net."""
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    h, w = image.shape[:2]

    img_r = cv2.resize(image_rgb, (256, 256)).astype(np.float32) / 255.0
    mean, std = np.array([0.485,0.456,0.406]), np.array([0.229,0.224,0.225])
    img_n = (img_r - mean) / std
    img_t = torch.tensor(img_n.transpose(2,0,1), dtype=torch.float32).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        mask = torch.sigmoid(model(img_t)).squeeze().cpu().numpy()

    mask_bin = (mask > 0.5).astype(np.uint8)
    mask_full = cv2.resize(mask_bin, (w, h), interpolation=cv2.INTER_NEAREST)

    wire_px = int(mask_full.sum())
    total_px = h * w
    return mask_full, {'wire_pixels': wire_px, 'total_pixels': total_px,
                       'wire_coverage': f'{wire_px/total_px*100:.2f}%'}

print('Inference helper functions defined.')

In [ ]:
def analyze_circuit(image_path):
    print(f'\n  Analyzing: {image_path}')
    print('  ' + '─' * 50)

    report = {'image': image_path, 'classification': None,
              'detections': None, 'segmentation': None}

    if inf_classifier is not None:
        print('\n  [1/3] Classifying components...')
        r = classify_component(inf_classifier, inf_class_info, image_path)
        report['classification'] = r
        print(f'        Prediction: {r["predicted_class"]} ({r["confidence"]})')

    if inf_detector is not None:
        print('\n  [2/3] Detecting IoT components...')
        dets = detect_components(inf_detector, image_path)
        report['detections'] = dets
        print(f'        Found {len(dets)} component(s)')
        for d in dets[:5]:
            print(f'          • {d["class"]} ({d["confidence"]})')
    wire_mask = None
    if inf_segmentor is not None:
        print('\n  [3/3] Segmenting wires...')
        wire_mask, seg_info = segment_wires(inf_segmentor, image_path)
        report['segmentation'] = seg_info
        print(f'        Wire coverage: {seg_info["wire_coverage"]}')

    return report, wire_mask

print('analyze_circuit() defined.')

In [ ]:
def visualize_analysis(image_path, report, wire_mask=None):
    """Create a visual analysis overlay."""
    image_rgb = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)

    n = 1 + (1 if report['detections'] else 0) + (1 if wire_mask is not None else 0)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 7))
    if n == 1: axes = [axes]
    idx = 0

    axes[idx].imshow(image_rgb)
    title = 'Input Image'
    if report['classification']:
        title += f'\nClassified: {report["classification"]["predicted_class"]}'
    axes[idx].set_title(title, fontsize=12); axes[idx].axis('off')

    if report['detections']:
        idx += 1
        axes[idx].imshow(image_rgb)
        for det in report['detections']:
            x1, y1, x2, y2 = det['bbox']
            rect = Rectangle((x1, y1), x2-x1, y2-y1,
                             linewidth=2, edgecolor='lime', facecolor='none')
            axes[idx].add_patch(rect)
            axes[idx].text(x1, y1-5, f'{det["class"]} {det["confidence"]}',
                           color='lime', fontsize=8, backgroundcolor='black')
        axes[idx].set_title('Component Detection', fontsize=12)
        axes[idx].axis('off')

    if wire_mask is not None:
        idx += 1
        overlay = image_rgb.copy()
        wire_color = np.array([0, 255, 100], dtype=np.uint8)
        mask_3d = np.stack([wire_mask]*3, axis=-1)
        overlay = np.where(mask_3d > 0,
                           (overlay*0.5 + wire_color*0.5).astype(np.uint8),
                           overlay)
        axes[idx].imshow(overlay)
        cov = report['segmentation']['wire_coverage'] if report['segmentation'] else 'N/A'
        axes[idx].set_title(f'Wire Segmentation\n(Coverage: {cov})', fontsize=12)
        axes[idx].axis('off')

    plt.suptitle('CircuitGPT — Circuit Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print('visualize_analysis() defined.')

In [ ]:
sample_dir = os.path.join(IMAGES_DIR, os.listdir(IMAGES_DIR)[1])
sample_images = [f for f in os.listdir(sample_dir)
                 if f.lower().endswith(('.jpg', '.png', '.jpeg'))]
test_image_path = os.path.join(sample_dir, sample_images[0])

print(f'Test image: {test_image_path}')

report, wire_mask = analyze_circuit(test_image_path)
visualize_analysis(test_image_path, report, wire_mask)

print('\nFull Report:')
print(json.dumps(report, indent=2))